# SGD + momentum on multi-MNIST 2-task with sparse hidden layer

Sanity-check classical SGD with momentum on the same architecture used by
the DEEP R progression. Masks `M1`/`M2` are frozen at their sparse init —
no pruning, no growth — so we get a clean read on what plain SGD+momentum
does on this sparse network. Loss is per-task softmax cross-entropy
summed over the 2 tasks (matches the rest of the progression). L1
shrinkage is applied outside the velocity (`-lr · l1 · sign(W)`).

Optimizer (drives the W update):

    v       <- momentum * v + (1 - momentum) * g
    W       <- (W - lr * v - lr * l1 * sign(W)) * M

In addition, two **diagnostic-only** EMAs run alongside, fully decoupled
from the optimizer momentum so we can choose plot-trace decay rates
independently of the gradient-descent step:

    v_aux   <- aux_momentum   * v_aux   + (1 - aux_momentum)   * g
    v_aux_2 <- aux_momentum_2 * v_aux_2 + (1 - aux_momentum_2) * g

Both `v_aux*` are unbiased EMAs of `g`, so their magnitudes are directly
comparable to each other without any `(1-β)` rescaling. Neither feeds
into the W update, so they cannot affect the loss. The plotted
comparison is between `v_aux` and `v_aux_2` — the optimizer's `v` is
not logged.

We log per chunk:
- mean `|v_aux|` and mean `|v_aux_2|`, factored by layer (W1 / W2) ×
  task relationship (within / cross), averaged over **active**
  connections.
- mean `|W|` in the same factorization.

Plus per-step `W`, `v_aux`, `v_aux_2`, `g` for a small set of tracked
weights (see the diagnostic-traces section at the bottom) so we can
study individual-weight dynamics — settling vs. hovering vs.
sign-flipping — and how the short/long aux EMAs respond to label
permutations.

## Setup

In [1]:
import os
import sys

REPO_ROOT = '/home/edan/local_projects/phd_research'
for p in (REPO_ROOT, os.path.join(REPO_ROOT, 'phd', 'structure_search')):
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from phd.jax_core.models import ltu

# Multi-MNIST 2-task layout
N_TASKS = 2
NUM_CLASSES = 10
INPUT_PER_TASK = 784
INPUT_DIM = INPUT_PER_TASK * N_TASKS              # 1568
OUTPUT_DIM = NUM_CLASSES * N_TASKS                # 20

# 2-layer LTU layout — each hidden unit hardwired to one output, equally split.
N_HIDDEN = 60                                     # must divide OUTPUT_DIM
HIDDEN_PER_OUTPUT = N_HIDDEN // OUTPUT_DIM        # 5
INPUT_FANIN = 128                                 # initial active input connections per hidden unit

assert N_HIDDEN % OUTPUT_DIM == 0, "N_HIDDEN must be divisible by OUTPUT_DIM"

print('JAX device:', jax.devices()[0])
print(f'INPUT_DIM={INPUT_DIM}  N_HIDDEN={N_HIDDEN}  OUTPUT_DIM={OUTPUT_DIM}'
      f'  HIDDEN_PER_OUTPUT={HIDDEN_PER_OUTPUT}  INPUT_FANIN={INPUT_FANIN}')

JAX device: cuda:0
INPUT_DIM=1568  N_HIDDEN=60  OUTPUT_DIM=20  HIDDEN_PER_OUTPUT=3  INPUT_FANIN=128


## Data

In [2]:
def load_data():
    """MNIST standardized per-pixel."""
    from data import load_dataset
    images, labels, _, _ = load_dataset('mnist', split='train')
    images = np.asarray(images, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int32)
    mean = images.mean(axis=0, keepdims=True)
    std = images.std(axis=0, keepdims=True)
    normalized = (images - mean) / np.maximum(std, 1e-3)
    return jnp.asarray(normalized), jnp.asarray(labels)


images, labels = load_data()
print('images:', images.shape, '   labels:', labels.shape)

images: (60000, 784)    labels: (60000,)


## Architecture

Forward: `z1 = x @ (W1 * M1); h = leaky_relu(z1); logits = h @ (W2 * M2)`.
Per-task softmax cross-entropy summed over 2 tasks. No biases.

In [3]:
def forward(W1, M1, W2, M2, x):
    z1 = x @ (W1 * M1)                              # (N_HIDDEN,)
    h = jax.nn.leaky_relu(z1)
    logits = h @ (W2 * M2)                          # (OUTPUT_DIM,)
    return logits, h, z1


def loss_fn(W1, M1, W2, M2, x, y):
    """Per-task softmax cross-entropy summed over 2 tasks."""
    logits, _, _ = forward(W1, M1, W2, M2, x)
    logits_pt = logits.reshape(N_TASKS, NUM_CLASSES)
    lp = jax.nn.log_softmax(logits_pt, axis=-1)
    return -jnp.mean(jnp.sum(jax.nn.one_hot(y, NUM_CLASSES) * lp, axis=-1))


def make_sample(images, labels, key):
    k1, k2 = jax.random.split(key)
    idx1 = jax.random.randint(k1, (), 0, images.shape[0])
    idx2 = jax.random.randint(k2, (), 0, images.shape[0])
    x = jnp.concatenate([images[idx1], images[idx2]])
    y = jnp.array([labels[idx1], labels[idx2]])
    return x, y

## Init

Per hidden unit: pick `INPUT_FANIN` random input pixels; one-hot to its assigned
output. Weights at active entries are Kaiming-uniform with the appropriate fan-in.
Future extensions (more outgoing connections, feature growth) just modify these
masks before passing them to the train functions.

In [4]:
def init_2layer_ltu(seed=0, n_hidden=N_HIDDEN, input_fanin=INPUT_FANIN):
    """Init W1, M1, W2, M2.

    M1: each column has `input_fanin` ones at random rows.
    M2: one-hot row, hidden unit i routes to output i // (n_hidden // OUTPUT_DIM).
    """
    k = jax.random.key(seed)
    k_m1, k_w1, k_w2 = jax.random.split(k, 3)

    # M1: per hidden unit, sample input_fanin random inputs.
    keys = jax.random.split(k_m1, n_hidden)

    def per_unit(key):
        noise = jax.random.uniform(key, (INPUT_DIM,))
        idx = jnp.argsort(-noise)[:input_fanin]
        return jnp.zeros(INPUT_DIM, dtype=jnp.int32).at[idx].set(1)

    M1_T = jax.vmap(per_unit)(keys)                              # (N_HIDDEN, INPUT_DIM)
    M1 = M1_T.T                                                  # (INPUT_DIM, N_HIDDEN)

    w1_bound = jnp.sqrt(3.0 / float(input_fanin))
    W1 = jax.random.uniform(k_w1, (INPUT_DIM, n_hidden),
                            minval=-w1_bound, maxval=w1_bound) * M1

    hidden_per_output = n_hidden // OUTPUT_DIM
    h2o = jnp.arange(n_hidden) // hidden_per_output              # (N_HIDDEN,)
    M2 = jax.nn.one_hot(h2o, OUTPUT_DIM, dtype=jnp.int32)        # (N_HIDDEN, OUTPUT_DIM)
    w2_bound = jnp.sqrt(3.0 / float(hidden_per_output))
    W2 = jax.random.uniform(k_w2, (n_hidden, OUTPUT_DIM),
                            minval=-w2_bound, maxval=w2_bound) * M2
    return W1, M1, W2, M2


# Sanity-check the init.
_W1, _M1, _W2, _M2 = init_2layer_ltu(seed=0)
print(f'M1 active per hidden unit: min={int(_M1.sum(0).min())} '
      f'mean={float(_M1.sum(0).mean())} max={int(_M1.sum(0).max())}')
print(f'M2 active per hidden unit: {int(_M2.sum(1).min())} (should be 1)')
print(f'M2 active per output: {int(_M2.sum(0).min())}/{int(_M2.sum(0).max())} '
      f'(should both be {HIDDEN_PER_OUTPUT})')

M1 active per hidden unit: min=128 mean=128.0 max=128
M2 active per hidden unit: 1 (should be 1)
M2 active per output: 3/3 (should both be 3)


## Training with SGD + momentum

Sparse-network forward, frozen masks, cross-entropy loss. Per step:

    v <- momentum * v + g            (only the gradient is momentumed)
    W <- (W - lr * v) * M            (mask re-applied so inactive
                                      entries stay zero)

Velocity is masked alongside W to keep inactive entries at v=0.

Snapshots track, per chunk, the per-(layer × within/cross)-category mean
of `|v|` and `|W|` over **active** connections, plus the final velocity
tensors `final_V1`/`final_V2` for end-of-training distribution plots.

In [5]:
# Within/cross-task masks for W1 (input × hidden) and W2 (hidden × output).
INPUT_TASK_J  = jnp.arange(INPUT_DIM) // INPUT_PER_TASK
HIDDEN_TASK_J = (jnp.arange(N_HIDDEN) // HIDDEN_PER_OUTPUT) // NUM_CLASSES
OUTPUT_TASK_J = jnp.arange(OUTPUT_DIM) // NUM_CLASSES
SAME_TASK_IH_J = (INPUT_TASK_J[:, None]  == HIDDEN_TASK_J[None, :])    # (IN, HIDDEN)
SAME_TASK_HO_J = (HIDDEN_TASK_J[:, None] == OUTPUT_TASK_J[None, :])    # (HIDDEN, OUT)


def train_sgd_momentum(W1_init, M1_init, W2_init, M2_init, images, labels, *,
                       lr=2**-6,
                       momentum=0.9,
                       aux_momentum=0.998,
                       aux_momentum_2=0.9998,
                       l1=0.0,
                       n_steps=500_000,
                       snapshot_every=2_000,
                       permute_period=0,
                       track_w1_idx=None,
                       track_w2_idx=None,
                       seed=0):
    """SGD with momentum + L1 + per-step weight tracking.

    The optimizer keeps a velocity `v` at β=`momentum` and uses it to
    update W:

        v       <- momentum     * v       + (1 - momentum)     * g
        W       <- (W - lr * v - lr * l1 * sign(W)) * M

    In addition, two **diagnostic-only** EMAs are tracked at
    `aux_momentum` and `aux_momentum_2` (decoupled from the optimizer):

        v_aux   <- aux_momentum   * v_aux   + (1 - aux_momentum)   * g
        v_aux_2 <- aux_momentum_2 * v_aux_2 + (1 - aux_momentum_2) * g

    Both `v_aux*` are unbiased EMAs of `g` so their magnitudes are
    directly comparable to each other. Neither enters the W update, so
    they cannot affect the loss. The plotted comparison is between
    `v_aux` and `v_aux_2` — the optimizer `v` is not logged.

    Snap-key mapping (kept for plot-code compatibility):
        mean_v_* / track_v* / final_V*       <- v_aux   (β=aux_momentum)
        mean_v_aux_* / track_v*_aux / final_V*_aux <- v_aux_2 (β=aux_momentum_2)
        momentum_main = aux_momentum,  momentum_aux = aux_momentum_2.

    Per-step tracking
    -----------------
    `track_w1_idx` / `track_w2_idx` are int arrays of shape (k, 2)
    listing (row, col) indices into W1 / W2 to log every step. For each
    tracked entry the post-step W, both EMA velocities, and the raw
    gradient are recorded. Output keys:

        track_w1, track_v1, track_v1_aux, track_g1  : (n_steps, k_w1)
        track_w2, track_v2, track_v2_aux, track_g2  : (n_steps, k_w2)

    plus `track_w1_idx`, `track_w2_idx` (np), and `permute_period`.
    Pass None (or omit) to disable tracking — empty (0, 2) is used."""
    n_chunks = n_steps // snapshot_every
    perm0_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    perm1_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)

    M1 = M1_init.astype(jnp.float32)
    M2 = M2_init.astype(jnp.float32)

    V1_init = jnp.zeros_like(W1_init)
    V2_init = jnp.zeros_like(W2_init)
    V1_aux_init  = jnp.zeros_like(W1_init)
    V2_aux_init  = jnp.zeros_like(W2_init)
    V1_aux2_init = jnp.zeros_like(W1_init)
    V2_aux2_init = jnp.zeros_like(W2_init)

    one_minus_main  = 1.0 - momentum
    one_minus_aux   = 1.0 - aux_momentum
    one_minus_aux_2 = 1.0 - aux_momentum_2
    lr_eff_main = lr

    if track_w1_idx is None:
        track_w1_idx = jnp.zeros((0, 2), dtype=jnp.int32)
    else:
        track_w1_idx = jnp.asarray(track_w1_idx, dtype=jnp.int32)
    if track_w2_idx is None:
        track_w2_idx = jnp.zeros((0, 2), dtype=jnp.int32)
    else:
        track_w2_idx = jnp.asarray(track_w2_idx, dtype=jnp.int32)
    track_w1_rows = track_w1_idx[:, 0]
    track_w1_cols = track_w1_idx[:, 1]
    track_w2_rows = track_w2_idx[:, 0]
    track_w2_cols = track_w2_idx[:, 1]

    init_carry = (W1_init, W2_init,
                  V1_init, V2_init,
                  V1_aux_init, V2_aux_init,
                  V1_aux2_init, V2_aux2_init,
                  perm0_init, perm1_init,
                  jnp.array(0, dtype=jnp.int32))

    cat_w1_within = M1 * SAME_TASK_IH_J.astype(jnp.float32)
    cat_w1_cross  = M1 * (~SAME_TASK_IH_J).astype(jnp.float32)
    cat_w2_within = M2 * SAME_TASK_HO_J.astype(jnp.float32)
    cat_w2_cross  = M2 * (~SAME_TASK_HO_J).astype(jnp.float32)

    def step_fn(carry, key):
        W1, W2, V1, V2, V1_aux, V2_aux, V1_aux2, V2_aux2, perm0, perm1, t = carry
        data_key, perm_key = jax.random.split(key)
        x, y_raw = make_sample(images, labels, data_key)
        y = jnp.array([perm0[y_raw[0]], perm1[y_raw[1]]])

        def _loss(W1_, W2_):
            return loss_fn(W1_, M1, W2_, M2, x, y)
        loss, (g1, g2) = jax.value_and_grad(_loss, argnums=(0, 1))(W1, W2)

        s1 = jnp.sign(W1)
        s2 = jnp.sign(W2)
        # Optimizer velocity (drives the W update).
        V1 = (momentum * V1 + one_minus_main * g1) * M1
        V2 = (momentum * V2 + one_minus_main * g2) * M2
        # Aux EMA #1 (diagnostic only, β=aux_momentum).
        V1_aux = (aux_momentum * V1_aux + one_minus_aux * g1) * M1
        V2_aux = (aux_momentum * V2_aux + one_minus_aux * g2) * M2
        # Aux EMA #2 (diagnostic only, β=aux_momentum_2, longer horizon).
        V1_aux2 = (aux_momentum_2 * V1_aux2 + one_minus_aux_2 * g1) * M1
        V2_aux2 = (aux_momentum_2 * V2_aux2 + one_minus_aux_2 * g2) * M2
        W1 = (W1 - lr_eff_main * V1 - lr * l1 * s1) * M1
        W2 = (W2 - lr_eff_main * V2 - lr * l1 * s2) * M2

        t_next = t + 1
        if permute_period > 0:
            should_perm = (t_next >= permute_period) & (t_next % permute_period == 0)
            # Deterministic: always permute task 0 so per-weight plots can
            # split tracked columns into permuted-task vs. unpermuted-task
            # without needing to read which task fired at runtime.
            new_perm = jax.random.permutation(perm_key, NUM_CLASSES).astype(jnp.int32)
            perm0 = jnp.where(should_perm, new_perm, perm0)

        # Per-step tracking (post-update). Note: track_v* stores the
        # FIRST aux EMA (β=aux_momentum) and track_v*_aux stores the
        # SECOND aux EMA (β=aux_momentum_2). Optimizer v is not logged.
        w1_tr     = W1[track_w1_rows, track_w1_cols]
        v1_tr     = V1_aux[track_w1_rows, track_w1_cols]
        v1_aux_tr = V1_aux2[track_w1_rows, track_w1_cols]
        g1_tr     = g1[track_w1_rows, track_w1_cols]
        w2_tr     = W2[track_w2_rows, track_w2_cols]
        v2_tr     = V2_aux[track_w2_rows, track_w2_cols]
        v2_aux_tr = V2_aux2[track_w2_rows, track_w2_cols]
        g2_tr     = g2[track_w2_rows, track_w2_cols]

        per_step = dict(
            loss=loss,
            track_w1=w1_tr, track_v1=v1_tr, track_v1_aux=v1_aux_tr, track_g1=g1_tr,
            track_w2=w2_tr, track_v2=v2_tr, track_v2_aux=v2_aux_tr, track_g2=g2_tr,
        )
        return (W1, W2, V1, V2, V1_aux, V2_aux,
                V1_aux2, V2_aux2, perm0, perm1, t_next), per_step

    def _masked_mean(arr, mask):
        return jnp.sum(arr * mask) / jnp.maximum(jnp.sum(mask), 1.0)

    def chunk_fn(carry, key):
        keys = jax.random.split(key, snapshot_every)
        carry, per_step = jax.lax.scan(step_fn, carry, keys)
        losses = per_step['loss']
        (W1_now, W2_now, _V1_opt, _V2_opt,
         V1_aux_now, V2_aux_now,
         V1_aux2_now, V2_aux2_now, _p0, _p1, t) = carry
        # `mean_v_*` snap keys map to V_aux (first diag EMA) and
        # `mean_v_aux_*` map to V_aux2 (second diag EMA). Optimizer
        # velocity is not logged.
        abs_V1     = jnp.abs(V1_aux_now)
        abs_V2     = jnp.abs(V2_aux_now)
        abs_V1_aux = jnp.abs(V1_aux2_now)
        abs_V2_aux = jnp.abs(V2_aux2_now)
        abs_W1 = jnp.abs(W1_now)
        abs_W2 = jnp.abs(W2_now)
        snap = dict(
            step=t,
            avg_loss=losses.mean(),
            mean_v_W1_within=_masked_mean(abs_V1, cat_w1_within),
            mean_v_W1_cross =_masked_mean(abs_V1, cat_w1_cross),
            mean_v_W2_within=_masked_mean(abs_V2, cat_w2_within),
            mean_v_W2_cross =_masked_mean(abs_V2, cat_w2_cross),
            mean_v_aux_W1_within=_masked_mean(abs_V1_aux, cat_w1_within),
            mean_v_aux_W1_cross =_masked_mean(abs_V1_aux, cat_w1_cross),
            mean_v_aux_W2_within=_masked_mean(abs_V2_aux, cat_w2_within),
            mean_v_aux_W2_cross =_masked_mean(abs_V2_aux, cat_w2_cross),
            mean_wL1_W1_within=_masked_mean(abs_W1, cat_w1_within),
            mean_wL1_W1_cross =_masked_mean(abs_W1, cat_w1_cross),
            mean_wL1_W2_within=_masked_mean(abs_W2, cat_w2_within),
            mean_wL1_W2_cross =_masked_mean(abs_W2, cat_w2_cross),
            track_w1=per_step['track_w1'],
            track_v1=per_step['track_v1'],
            track_v1_aux=per_step['track_v1_aux'],
            track_g1=per_step['track_g1'],
            track_w2=per_step['track_w2'],
            track_v2=per_step['track_v2'],
            track_v2_aux=per_step['track_v2_aux'],
            track_g2=per_step['track_g2'],
        )
        return carry, snap

    rng = jax.random.key(seed)
    chunk_keys = jax.random.split(rng, n_chunks)
    final_carry, snaps = jax.lax.scan(chunk_fn, init_carry, chunk_keys)
    snaps = {k: jax.device_get(v) for k, v in snaps.items()}
    snaps['final_W1'] = jax.device_get(final_carry[0])
    snaps['final_W2'] = jax.device_get(final_carry[1])
    # final_V* stores V_aux (first diag); final_V*_aux stores V_aux2.
    snaps['final_V1']     = jax.device_get(final_carry[4])
    snaps['final_V2']     = jax.device_get(final_carry[5])
    snaps['final_V1_aux'] = jax.device_get(final_carry[6])
    snaps['final_V2_aux'] = jax.device_get(final_carry[7])
    snaps['momentum_main'] = float(aux_momentum)
    snaps['momentum_aux']  = float(aux_momentum_2)
    # Flatten per-step tracked traces from (n_chunks, snapshot_every, k)
    # to (n_steps, k) so they're indexed by absolute step.
    for k in ('track_w1', 'track_v1', 'track_v1_aux', 'track_g1',
              'track_w2', 'track_v2', 'track_v2_aux', 'track_g2'):
        arr = snaps[k]
        snaps[k] = arr.reshape(arr.shape[0] * arr.shape[1], arr.shape[2])
    # NOTE: track_w*_idx and permute_period are attached by the caller —
    # np.asarray on a tracer would explode under jit, and the caller has
    # them in scope already.
    return snaps

## Baseline (plain SGD, frozen masks)

Same init, plain SGD (no momentum) with frozen masks — the no-momentum
reference loss curve.

In [6]:
def train_baseline(W1_init, M1_init, W2_init, M2_init, images, labels, *,
                   lr=2**-7,
                   momentum=0.0,
                   n_steps=500_000,
                   snapshot_every=2_000,
                   permute_period=0,
                   seed=0):
    """SGD (optionally with momentum) with frozen masks. No L1, no noise.

    Defaults to plain SGD (`momentum=0.0`) so this is a clean
    no-momentum reference for the SGD+momentum run."""
    n_chunks = n_steps // snapshot_every
    perm0_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    perm1_init = jnp.arange(NUM_CLASSES, dtype=jnp.int32)
    V1_init = jnp.zeros_like(W1_init)
    V2_init = jnp.zeros_like(W2_init)
    init_carry = (W1_init, W2_init, V1_init, V2_init,
                  perm0_init, perm1_init,
                  jnp.array(0, dtype=jnp.int32))

    def step_fn(carry, key):
        W1, W2, V1, V2, perm0, perm1, t = carry
        data_key, perm_key = jax.random.split(key)
        x, y_raw = make_sample(images, labels, data_key)
        y = jnp.array([perm0[y_raw[0]], perm1[y_raw[1]]])
        def _loss(W1_, W2_):
            return loss_fn(W1_, M1_init, W2_, M2_init, x, y)
        loss, (g1, g2) = jax.value_and_grad(_loss, argnums=(0, 1))(W1, W2)
        V1 = momentum * V1 + g1
        V2 = momentum * V2 + g2
        W1 = (W1 - lr * V1) * M1_init
        W2 = (W2 - lr * V2) * M2_init
        t_next = t + 1
        if permute_period > 0:
            should_perm = (t_next >= permute_period) & (t_next % permute_period == 0)
            pk1, pk2 = jax.random.split(perm_key)
            which = jax.random.randint(pk1, (), 0, N_TASKS)
            new_perm = jax.random.permutation(pk2, NUM_CLASSES).astype(jnp.int32)
            perm0 = jnp.where(should_perm & (which == 0), new_perm, perm0)
            perm1 = jnp.where(should_perm & (which == 1), new_perm, perm1)
        return (W1, W2, V1, V2, perm0, perm1, t_next), loss

    def chunk_fn(carry, key):
        keys = jax.random.split(key, snapshot_every)
        carry, losses = jax.lax.scan(step_fn, carry, keys)
        W1, W2, _V1, _V2, _p0, _p1, t = carry
        return carry, dict(step=t, avg_loss=losses.mean())

    rng = jax.random.key(seed)
    chunk_keys = jax.random.split(rng, n_chunks)
    final_carry, snaps = jax.lax.scan(chunk_fn, init_carry, chunk_keys)
    snaps = {k: jax.device_get(v) for k, v in snaps.items()}
    snaps['final_W1'] = jax.device_get(final_carry[0])
    snaps['final_W2'] = jax.device_get(final_carry[1])
    return snaps

## Run

`SGD_CONFIG.momentum=0.9` is the headline run. The baseline reuses the
same lr but with `momentum=0.0`, so the loss panel directly contrasts
momentum vs. plain SGD on identical inits.

In [7]:
# Shared init (same seed -> same starting topology and weights for both runs).
W1_init, M1_init, W2_init, M2_init = init_2layer_ltu(seed=0)


def select_track_indices(M1, M2, *, n_w2=4, n_w1_within=4, n_w1_cross=4,
                          permuted_task=0, seed=0):
    """Stratified sample of (row, col) indices for per-step tracking.
    Within each category the first half of picks come from hidden units
    in `permuted_task` and the second half from the other task — so the
    diagnostic plots' first 2 columns show permuted-task features and
    the last 2 show unpermuted-task features. n_w2 / n_w1_within /
    n_w1_cross must be even."""
    assert n_w2 % 2 == 0 and n_w1_within % 2 == 0 and n_w1_cross % 2 == 0, (
        'n_w2/n_w1_within/n_w1_cross must be even (split per task)')
    rng = np.random.default_rng(seed)
    M1_np = np.asarray(M1).astype(bool)
    M2_np = np.asarray(M2).astype(bool)
    input_task  = np.arange(INPUT_DIM)  // INPUT_PER_TASK
    hidden_task = (np.arange(N_HIDDEN) // HIDDEN_PER_OUTPUT) // NUM_CLASSES
    same_ih = input_task[:, None] == hidden_task[None, :]

    perm_t  = int(permuted_task)
    other_t = 1 - perm_t
    h_perm  = np.where(hidden_task == perm_t)[0]
    h_other = np.where(hidden_task == other_t)[0]

    def pick(active_2d, hidden_subset, k, hidden_axis):
        # Filter rows of `active_2d` (shape (N_active, 2)) by whether the
        # entry's hidden-unit index (column `hidden_axis`) is in `hidden_subset`.
        mask = np.isin(active_2d[:, hidden_axis], hidden_subset)
        sub  = active_2d[mask]
        idx  = rng.choice(len(sub), size=k, replace=False)
        return sub[idx]

    # M2 has shape (N_HIDDEN, OUTPUT_DIM)  -> argwhere col 0 is hidden.
    # M1 has shape (INPUT_DIM, N_HIDDEN)   -> argwhere col 1 is hidden.
    w2_active        = np.argwhere(M2_np)
    w1_within_active = np.argwhere(M1_np & same_ih)
    w1_cross_active  = np.argwhere(M1_np & ~same_ih)

    w2_pick        = np.concatenate([
        pick(w2_active,        h_perm,  n_w2 // 2,        hidden_axis=0),
        pick(w2_active,        h_other, n_w2 // 2,        hidden_axis=0),
    ], axis=0)
    w1_within_pick = np.concatenate([
        pick(w1_within_active, h_perm,  n_w1_within // 2, hidden_axis=1),
        pick(w1_within_active, h_other, n_w1_within // 2, hidden_axis=1),
    ], axis=0)
    w1_cross_pick  = np.concatenate([
        pick(w1_cross_active,  h_perm,  n_w1_cross // 2,  hidden_axis=1),
        pick(w1_cross_active,  h_other, n_w1_cross // 2,  hidden_axis=1),
    ], axis=0)

    track_w1_idx = np.concatenate([w1_within_pick, w1_cross_pick],
                                  axis=0).astype(np.int32)
    track_w2_idx = w2_pick.astype(np.int32)
    w1_categories = np.array(['within'] * n_w1_within + ['cross'] * n_w1_cross)
    w2_categories = np.array(['within'] * n_w2)
    return dict(
        track_w1_idx=track_w1_idx,
        track_w2_idx=track_w2_idx,
        w1_categories=w1_categories,
        w2_categories=w2_categories,
        n_w1_within=n_w1_within,
        n_w1_cross=n_w1_cross,
        n_w2=n_w2,
        permuted_task=perm_t,
    )


TRACK = select_track_indices(M1_init, M2_init,
                             n_w2=4, n_w1_within=4, n_w1_cross=4,
                             permuted_task=0, seed=42)
print(f"Permuted task = {TRACK['permuted_task']}; first 2 cols of each "
      "diagnostic plot show permuted-task features.")
print('Tracked W2 (hidden, output):', TRACK['track_w2_idx'].tolist())
print('Tracked W1 within (input, hidden):',
      TRACK['track_w1_idx'][:TRACK['n_w1_within']].tolist())
print('Tracked W1 cross  (input, hidden):',
      TRACK['track_w1_idx'][TRACK['n_w1_within']:].tolist())

SGD_CONFIG = dict(
    lr=2**-6,
    momentum=0.9,            # optimizer β — drives W update
    aux_momentum=0.998,      # diag EMA #1 (shorter horizon)
    aux_momentum_2=0.9998,   # diag EMA #2 (longer horizon)
    l1=1e-4,
    n_steps=110_000,
    snapshot_every=2_000,
    permute_period=100_000,
    seed=0,
)
BASELINE_LR = 2**-6

train_sgd_jit = jax.jit(
    train_sgd_momentum,
    static_argnames=('lr', 'momentum', 'aux_momentum', 'aux_momentum_2',
                     'l1', 'n_steps', 'snapshot_every',
                     'permute_period', 'seed'),
)
train_baseline_jit = jax.jit(
    train_baseline,
    static_argnames=('lr', 'momentum', 'n_steps', 'snapshot_every',
                     'permute_period', 'seed'),
)

print('\nRunning SGD + momentum...')
sgd_snaps = train_sgd_jit(W1_init, M1_init, W2_init, M2_init,
                          images, labels,
                          track_w1_idx=jnp.asarray(TRACK['track_w1_idx']),
                          track_w2_idx=jnp.asarray(TRACK['track_w2_idx']),
                          **SGD_CONFIG)
sgd_snaps['track_w1_idx']   = TRACK['track_w1_idx']
sgd_snaps['track_w2_idx']   = TRACK['track_w2_idx']
sgd_snaps['permute_period'] = int(SGD_CONFIG['permute_period'])
print(f'  final loss: {float(sgd_snaps["avg_loss"][-1]):.4f}')
print(f'  final diag |v|  β={SGD_CONFIG["aux_momentum"]:.4f}'
      f'  W1 within / cross: '
      f'{float(sgd_snaps["mean_v_W1_within"][-1]):.3e} '
      f'/ {float(sgd_snaps["mean_v_W1_cross"][-1]):.3e}')
print(f'  final diag |v|  β={SGD_CONFIG["aux_momentum_2"]:.4f}'
      f'  W1 within / cross: '
      f'{float(sgd_snaps["mean_v_aux_W1_within"][-1]):.3e} '
      f'/ {float(sgd_snaps["mean_v_aux_W1_cross"][-1]):.3e}')
print(f'  final diag |v|  β={SGD_CONFIG["aux_momentum"]:.4f}'
      f'  W2 within / cross: '
      f'{float(sgd_snaps["mean_v_W2_within"][-1]):.3e} '
      f'/ {float(sgd_snaps["mean_v_W2_cross"][-1]):.3e}')
print(f'  final diag |v|  β={SGD_CONFIG["aux_momentum_2"]:.4f}'
      f'  W2 within / cross: '
      f'{float(sgd_snaps["mean_v_aux_W2_within"][-1]):.3e} '
      f'/ {float(sgd_snaps["mean_v_aux_W2_cross"][-1]):.3e}')
# Reference noise-floor for |v_short|/|v_long| (the two diag EMAs) under
# i.i.d. gradients.
_bm, _ba = SGD_CONFIG['aux_momentum'], SGD_CONFIG['aux_momentum_2']
_noise_ratio = float(np.sqrt((1 - _bm) * (1 + _ba) / ((1 + _bm) * (1 - _ba))))
print(f'  i.i.d. noise-floor ratio |v_short|/|v_long| ≈ {_noise_ratio:.3f}'
      f'  (signal-dominated regime ≈ 1.0)')

print('\nRunning baseline (plain SGD, no momentum)...')
baseline_snaps = train_baseline_jit(W1_init, M1_init, W2_init, M2_init, images, labels,
                                     lr=BASELINE_LR,
                                     momentum=0.0,
                                     n_steps=SGD_CONFIG['n_steps'],
                                     snapshot_every=SGD_CONFIG['snapshot_every'],
                                     permute_period=SGD_CONFIG['permute_period'],
                                     seed=SGD_CONFIG['seed'])
print(f'  final loss: {float(baseline_snaps["avg_loss"][-1]):.4f}')

Permuted task = 0; first 2 cols of each diagnostic plot show permuted-task features.
Tracked W2 (hidden, output): [[2, 0], [23, 7], [42, 14], [59, 19]]
Tracked W1 within (input, hidden): [[558, 7], [69, 0], [854, 53], [1192, 48]]
Tracked W1 cross  (input, hidden): [[1360, 19], [1381, 20], [407, 49], [613, 31]]

Running SGD + momentum...
  final loss: 0.7193
  final diag |v|  β=0.9980  W1 within / cross: 7.541e-04 / 8.584e-04
  final diag |v|  β=0.9998  W1 within / cross: 3.678e-04 / 2.605e-04
  final diag |v|  β=0.9980  W2 within / cross: 2.935e-03 / 0.000e+00
  final diag |v|  β=0.9998  W2 within / cross: 1.744e-03 / 0.000e+00
  i.i.d. noise-floor ratio |v_short|/|v_long| ≈ 3.164  (signal-dominated regime ≈ 1.0)

Running baseline (plain SGD, no momentum)...
  final loss: 0.8215


## Velocity traces

Mean per-weight `|v|` at chunk boundaries, factored by layer (W1 / W2)
× task relation (within / cross). Within-task entries should sustain
larger `|v|` once training is rolling, since their gradients line up
in a consistent direction; cross-task entries should average down toward
zero.

Note: with the current architecture each hidden unit is hardwired to a
single within-task output, so `M2` has no cross-task active entries and
the W2-cross trace is identically zero by construction.

In [8]:
def velocity_traces(snaps):
    """EMA `|v|` traces for both the main and aux decays."""
    return dict(
        steps=np.asarray(snaps['step']),
        momentum_main=float(snaps['momentum_main']),
        momentum_aux =float(snaps['momentum_aux']),
        # main (β = momentum_main)
        W1_within=np.asarray(snaps['mean_v_W1_within']),
        W1_cross =np.asarray(snaps['mean_v_W1_cross']),
        W2_within=np.asarray(snaps['mean_v_W2_within']),
        W2_cross =np.asarray(snaps['mean_v_W2_cross']),
        # aux (β = momentum_aux)
        W1_within_aux=np.asarray(snaps['mean_v_aux_W1_within']),
        W1_cross_aux =np.asarray(snaps['mean_v_aux_W1_cross']),
        W2_within_aux=np.asarray(snaps['mean_v_aux_W2_within']),
        W2_cross_aux =np.asarray(snaps['mean_v_aux_W2_cross']),
    )


def w_l1_traces(snaps):
    return dict(
        steps=np.asarray(snaps['step']),
        W1_within=np.asarray(snaps['mean_wL1_W1_within']),
        W1_cross =np.asarray(snaps['mean_wL1_W1_cross']),
        W2_within=np.asarray(snaps['mean_wL1_W2_within']),
        W2_cross =np.asarray(snaps['mean_wL1_W2_cross']),
    )


vels = velocity_traces(sgd_snaps)
wl1s = w_l1_traces(sgd_snaps)

bm = vels['momentum_main']
ba = vels['momentum_aux']
print(f'final |v|  β={bm:.4f}  W1 within / cross: '
      f'{vels["W1_within"][-1]:.3e} / {vels["W1_cross"][-1]:.3e}')
print(f'final |v|  β={ba:.4f}  W1 within / cross: '
      f'{vels["W1_within_aux"][-1]:.3e} / {vels["W1_cross_aux"][-1]:.3e}')
print(f'final |v|  β={bm:.4f}  W2 within / cross: '
      f'{vels["W2_within"][-1]:.3e} / {vels["W2_cross"][-1]:.3e}')
print(f'final |v|  β={ba:.4f}  W2 within / cross: '
      f'{vels["W2_within_aux"][-1]:.3e} / {vels["W2_cross_aux"][-1]:.3e}')
print(f'final mean |W|     W1 within / cross: '
      f'{wl1s["W1_within"][-1]:.3e} / {wl1s["W1_cross"][-1]:.3e}')
print(f'final mean |W|     W2 within / cross: '
      f'{wl1s["W2_within"][-1]:.3e} / {wl1s["W2_cross"][-1]:.3e}')

final |v|  β=0.9980  W1 within / cross: 7.541e-04 / 8.584e-04
final |v|  β=0.9998  W1 within / cross: 3.678e-04 / 2.605e-04
final |v|  β=0.9980  W2 within / cross: 2.935e-03 / 0.000e+00
final |v|  β=0.9998  W2 within / cross: 1.744e-03 / 0.000e+00
final mean |W|     W1 within / cross: 1.933e-01 / 6.178e-02
final mean |W|     W2 within / cross: 1.110e+00 / 0.000e+00


## Plots

In [9]:
WITHIN_COLOR = '#1f77b4'   # blue
CROSS_COLOR  = '#d62728'   # red


def plot_loss(sgd_snaps, baseline_snaps):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=np.asarray(sgd_snaps['step']),
                             y=np.asarray(sgd_snaps['avg_loss']),
                             mode='lines', name='SGD + momentum'))
    fig.add_trace(go.Scatter(x=np.asarray(baseline_snaps['step']),
                             y=np.asarray(baseline_snaps['avg_loss']),
                             mode='lines', name='baseline SGD',
                             line=dict(dash='dot')))
    fig.update_layout(title='Loss over training',
                      xaxis_title='step',
                      yaxis_title='mean loss over snapshot',
                      width=900, height=420)
    fig.show()
    return fig


def _plot_2panel_traces(traces, *, title, ylabel, shared_yaxes=True, log_y=False):
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=('W1 (incoming)', 'W2 (outgoing)'),
                        shared_yaxes=shared_yaxes)
    s = traces['steps']
    fig.add_trace(go.Scatter(x=s, y=traces['W1_within'], mode='lines',
                             name='W1 within',
                             line=dict(color=WITHIN_COLOR)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=s, y=traces['W1_cross'],  mode='lines',
                             name='W1 cross',
                             line=dict(color=CROSS_COLOR)),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=s, y=traces['W2_within'], mode='lines',
                             name='W2 within',
                             line=dict(color=WITHIN_COLOR, dash='dash')),
                  row=1, col=2)
    fig.add_trace(go.Scatter(x=s, y=traces['W2_cross'],  mode='lines',
                             name='W2 cross',
                             line=dict(color=CROSS_COLOR, dash='dash')),
                  row=1, col=2)
    if log_y:
        fig.update_yaxes(type='log')
    fig.update_xaxes(title_text='step', row=1, col=1)
    fig.update_xaxes(title_text='step', row=1, col=2)
    fig.update_yaxes(title_text=ylabel, row=1, col=1)
    fig.update_layout(title=title, width=1000, height=420)
    fig.show()
    return fig


def plot_velocity_traces(traces):
    """EMA velocity magnitude `|v|` per layer × task relation × decay.
    Solid lines: main β. Dashed lines: aux β."""
    bm = traces['momentum_main']
    ba = traces['momentum_aux']
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=('W1 (incoming)', 'W2 (outgoing)'),
                        shared_yaxes=False)
    s = traces['steps']

    def _add(panel, key, color, dash, name):
        fig.add_trace(go.Scatter(x=s, y=traces[key], mode='lines',
                                 name=name,
                                 line=dict(color=color, dash=dash)),
                      row=1, col=panel)

    _add(1, 'W1_within',     WITHIN_COLOR, 'solid', f'W1 within  β={bm:.4f}')
    _add(1, 'W1_cross',      CROSS_COLOR,  'solid', f'W1 cross   β={bm:.4f}')
    _add(1, 'W1_within_aux', WITHIN_COLOR, 'dash',  f'W1 within  β={ba:.4f}')
    _add(1, 'W1_cross_aux',  CROSS_COLOR,  'dash',  f'W1 cross   β={ba:.4f}')
    _add(2, 'W2_within',     WITHIN_COLOR, 'solid', f'W2 within  β={bm:.4f}')
    _add(2, 'W2_cross',      CROSS_COLOR,  'solid', f'W2 cross   β={bm:.4f}')
    _add(2, 'W2_within_aux', WITHIN_COLOR, 'dash',  f'W2 within  β={ba:.4f}')
    _add(2, 'W2_cross_aux',  CROSS_COLOR,  'dash',  f'W2 cross   β={ba:.4f}')

    fig.update_yaxes(type='log')
    fig.update_xaxes(title_text='step', row=1, col=1)
    fig.update_xaxes(title_text='step', row=1, col=2)
    fig.update_yaxes(title_text='|v|', row=1, col=1)
    fig.update_layout(title='EMA velocity magnitude |v| over active connections',
                      width=1100, height=440)
    fig.show()
    return fig


def plot_w_l1_traces(traces):
    """Mean per-weight |W|, split by layer × task relation."""
    return _plot_2panel_traces(
        traces,
        title='Mean |W| over active connections',
        ylabel='mean |W|',
        shared_yaxes=False, log_y=False,
    )


def _category_indices(M1, M2):
    M1_b = np.asarray(M1).astype(bool)
    M2_b = np.asarray(M2).astype(bool)
    same_ih = np.asarray(SAME_TASK_IH_J)
    same_ho = np.asarray(SAME_TASK_HO_J)
    return dict(
        W1_within=M1_b & same_ih,
        W1_cross =M1_b & ~same_ih,
        W2_within=M2_b & same_ho,
        W2_cross =M2_b & ~same_ho,
    )


def _add_hist_pair(fig, vals_within, vals_cross, *, row, col, nbins, name_prefix):
    """Overlaid within/cross histograms with shared bin edges per panel."""
    vals_within = np.asarray(vals_within).ravel()
    vals_cross  = np.asarray(vals_cross).ravel()
    combined = np.concatenate([vals_within, vals_cross])
    if combined.size == 0:
        return
    lo = float(np.min(combined))
    hi = float(np.max(combined))
    if hi <= lo:
        hi = lo + 1.0
    size = (hi - lo) / nbins
    # Nudge `end` past `hi` so the max sample isn't dropped from rounding.
    xbins = dict(start=lo, end=hi + size * 0.5, size=size)
    if vals_within.size > 0:
        fig.add_trace(go.Histogram(x=vals_within,
                                    name=f'{name_prefix} within',
                                    marker_color=WITHIN_COLOR, opacity=0.6,
                                    xbins=xbins, autobinx=False),
                      row=row, col=col)
    if vals_cross.size > 0:
        fig.add_trace(go.Histogram(x=vals_cross,
                                    name=f'{name_prefix} cross',
                                    marker_color=CROSS_COLOR, opacity=0.6,
                                    xbins=xbins, autobinx=False),
                      row=row, col=col)


def plot_velocity_distribution(sgd_snaps, M1, M2, nbins=60):
    """Final-step distribution of signed EMA velocity v for each
    (layer × decay) panel."""
    cats = _category_indices(M1, M2)
    final_V1     = np.asarray(sgd_snaps['final_V1'])
    final_V2     = np.asarray(sgd_snaps['final_V2'])
    final_V1_aux = np.asarray(sgd_snaps['final_V1_aux'])
    final_V2_aux = np.asarray(sgd_snaps['final_V2_aux'])
    bm = float(sgd_snaps['momentum_main'])
    ba = float(sgd_snaps['momentum_aux'])

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            f'W1 (incoming)  β={bm:.4f}',
            f'W2 (outgoing)  β={bm:.4f}',
            f'W1 (incoming)  β={ba:.4f}',
            f'W2 (outgoing)  β={ba:.4f}',
        ),
    )

    def _panel(arr, layer, row, col):
        wkey = 'W1_within' if layer == 'W1' else 'W2_within'
        ckey = 'W1_cross'  if layer == 'W1' else 'W2_cross'
        _add_hist_pair(
            fig,
            arr[cats[wkey]] if cats[wkey].any() else np.array([]),
            arr[cats[ckey]] if cats[ckey].any() else np.array([]),
            row=row, col=col, nbins=nbins, name_prefix=f'{layer} β={bm:.4f}' if row == 1 else f'{layer} β={ba:.4f}',
        )

    _panel(final_V1,     'W1', row=1, col=1)
    _panel(final_V2,     'W2', row=1, col=2)
    _panel(final_V1_aux, 'W1', row=2, col=1)
    _panel(final_V2_aux, 'W2', row=2, col=2)

    for r in (1, 2):
        for c in (1, 2):
            fig.update_xaxes(title_text='v', row=r, col=c)
    fig.update_yaxes(title_text='count', row=1, col=1)
    fig.update_yaxes(title_text='count', row=2, col=1)
    fig.update_layout(title='Final EMA velocity v distribution',
                      barmode='overlay',
                      width=1100, height=720)
    fig.show()
    return fig


def plot_w_l1_distribution(sgd_snaps, M1, M2, nbins=60):
    """Final-step distribution of |W| for each category."""
    cats = _category_indices(M1, M2)
    final_W1 = np.abs(np.asarray(sgd_snaps['final_W1']))
    final_W2 = np.abs(np.asarray(sgd_snaps['final_W2']))
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=('W1 (incoming)', 'W2 (outgoing)'))
    _add_hist_pair(
        fig,
        final_W1[cats['W1_within']] if cats['W1_within'].any() else np.array([]),
        final_W1[cats['W1_cross']]  if cats['W1_cross'].any()  else np.array([]),
        row=1, col=1, nbins=nbins, name_prefix='W1',
    )
    _add_hist_pair(
        fig,
        final_W2[cats['W2_within']] if cats['W2_within'].any() else np.array([]),
        final_W2[cats['W2_cross']]  if cats['W2_cross'].any()  else np.array([]),
        row=1, col=2, nbins=nbins, name_prefix='W2',
    )
    fig.update_xaxes(title_text='|W|', row=1, col=1)
    fig.update_xaxes(title_text='|W|', row=1, col=2)
    fig.update_yaxes(title_text='count', row=1, col=1)
    fig.update_layout(title='Final |W| distribution',
                      barmode='overlay',
                      width=1000, height=420)
    fig.show()
    return fig

In [10]:
plot_loss(sgd_snaps, baseline_snaps)
plot_velocity_traces(vels)
plot_w_l1_traces(wl1s)
plot_velocity_distribution(sgd_snaps, M1_init, M2_init)
plot_w_l1_distribution(sgd_snaps, M1_init, M2_init);

## Per-weight diagnostic traces

For each of the 12 tracked weights (4 W2-within, 4 W1-within, 4 W1-cross),
plot per-step:

1. **W**: the weight value over training. Lets us see "settles to non-zero",
   "hovers near 0", or "oscillates / sign-flips".
2. **signed v_short, v_long**: the two EMA velocities. Both have the same
   E[v] = E[g], so they're directly comparable. v_short reacts faster to
   recent gradient direction; v_long is the longer-window estimator.
3. **|v_short| / |v_long|**: the discriminator. Reference lines:
   - dotted gray at the i.i.d. noise-floor ratio (≈3.24 for β=0.9 vs 0.99)
     — sustained values near here mean the gradient is noise-dominated.
   - solid gray at 1 — sustained values near here mean a consistent
     gradient signal.
   - **transient spikes well above the noise floor** are the signature of
     a sign flip: v_short jumps to the new direction while v_long is
     still cancelling old + new and stays small.

Vertical red dashed lines mark permutation events (none if
`permute_period=0` — change the config and re-run to see post-permutation
dynamics).

In [11]:
def _perm_steps(snaps):
    """Step indices (1-indexed) at which a permutation fired."""
    period = int(snaps['permute_period'])
    n_steps = int(np.asarray(snaps['track_w2']).shape[0]) if 'track_w2' in snaps else 0
    if period <= 0 or n_steps == 0:
        return np.array([], dtype=np.int64)
    return np.arange(period, n_steps + 1, period, dtype=np.int64)


def _noise_ratio(bm, ba):
    return float(np.sqrt((1 - bm) * (1 + ba) / ((1 + bm) * (1 - ba))))


def plot_tracked_category(snaps, *, layer_letter, slot_indices, idx_pairs,
                           category_label, downsample=1,
                           step_window=None, n_permuted_cols=None):
    """layer_letter: '1' (W1) or '2' (W2). slot_indices: positions into
    the per-step tracked array. idx_pairs: list of (row, col) for
    subplot titles. `downsample` keeps every Nth step in the plot to
    cap browser cost on long runs. `step_window=(start, end)` (1-indexed,
    inclusive) restricts the x-range to that slice of steps.
    `n_permuted_cols`: if set, the first that many subplot titles are
    tagged 'PERM' and the remainder 'OTHER'."""
    Wt = np.asarray(snaps[f'track_w{layer_letter}'])
    Vm = np.asarray(snaps[f'track_v{layer_letter}'])
    Va = np.asarray(snaps[f'track_v{layer_letter}_aux'])
    n_steps = Wt.shape[0]
    if step_window is not None:
        s0, s1 = int(step_window[0]), int(step_window[1])
        s0 = max(1, s0); s1 = min(n_steps, s1)
        sl = slice(s0 - 1, s1)
        Wt = Wt[sl]; Vm = Vm[sl]; Va = Va[sl]
        step_idx = np.arange(s0, s1 + 1)
    else:
        step_idx = np.arange(1, n_steps + 1)
    if downsample > 1:
        step_idx = step_idx[::downsample]
    bm = float(snaps['momentum_main'])
    ba = float(snaps['momentum_aux'])
    noise_floor = _noise_ratio(bm, ba)
    perm_steps = _perm_steps(snaps)

    n = len(slot_indices)
    fig = make_subplots(
        rows=3, cols=n,
        shared_xaxes=True, vertical_spacing=0.06, horizontal_spacing=0.05,
        subplot_titles=[
            f'({i},{j}) {("PERM" if n_permuted_cols is not None and k < n_permuted_cols else "OTHER") if n_permuted_cols is not None else ""}'.strip()
            for k, (i, j) in enumerate(idx_pairs)
        ],
        row_heights=[0.30, 0.35, 0.35],
    )
    eps = 1e-12
    for c, slot in enumerate(slot_indices, start=1):
        W_s  = Wt[:, slot][::downsample] if downsample > 1 else Wt[:, slot]
        Vm_s = Vm[:, slot][::downsample] if downsample > 1 else Vm[:, slot]
        Va_s = Va[:, slot][::downsample] if downsample > 1 else Va[:, slot]
        ratio = np.abs(Vm_s) / (np.abs(Va_s) + eps)

        fig.add_trace(go.Scatter(x=step_idx, y=W_s, mode='lines',
                                 name='W', showlegend=(c == 1),
                                 line=dict(color='black', width=1.2)),
                      row=1, col=c)
        fig.add_hline(y=0, line=dict(color='gray', dash='dot', width=1),
                      row=1, col=c)

        fig.add_trace(go.Scatter(x=step_idx, y=Vm_s, mode='lines',
                                 name=f'v (β={bm:.4f})',
                                 showlegend=(c == 1),
                                 line=dict(color=WITHIN_COLOR, width=1)),
                      row=2, col=c)
        fig.add_trace(go.Scatter(x=step_idx, y=Va_s, mode='lines',
                                 name=f'v (β={ba:.4f})',
                                 showlegend=(c == 1),
                                 line=dict(color=CROSS_COLOR, width=1.4)),
                      row=2, col=c)
        fig.add_hline(y=0, line=dict(color='gray', dash='dot', width=1),
                      row=2, col=c)

        fig.add_trace(go.Scatter(x=step_idx, y=ratio, mode='lines',
                                 name='|v_s|/|v_l|', showlegend=(c == 1),
                                 line=dict(color='purple', width=1)),
                      row=3, col=c)
        fig.add_hline(y=noise_floor,
                      line=dict(color='gray', dash='dot', width=1),
                      row=3, col=c)
        fig.add_hline(y=1.0,
                      line=dict(color='gray', dash='dash', width=1),
                      row=3, col=c)
        fig.update_yaxes(type='log', row=3, col=c)

        for ps in perm_steps:
            if int(ps) < int(step_idx[0]) or int(ps) > int(step_idx[-1]):
                continue
            fig.add_vline(x=int(ps),
                          line=dict(color='red', dash='dot', width=1),
                          row='all', col=c)

    fig.update_yaxes(title_text='W', row=1, col=1)
    fig.update_yaxes(title_text='v (signed)', row=2, col=1)
    fig.update_yaxes(title_text='|v_s|/|v_l|', row=3, col=1)
    for c in range(1, n + 1):
        fig.update_xaxes(title_text='step', row=3, col=c)
    fig.update_layout(
        title=f'{category_label}  '
              f'(noise-floor ratio ≈ {noise_floor:.2f}, signal ≈ 1.0)',
        width=320 * n, height=760,
    )
    return fig


def plot_post_perm_ratio_hist(snaps, *, layer_letter, slot_indices,
                               category_label, window=2_000, nbins=60):
    """Histogram of log10(|v_short|/|v_long|) in three time windows:
    early (first `window`), settled (last `window` before first perm or
    end of run), and post-perm (first `window` after each perm event,
    pooled). If permute_period=0, only early + settled are drawn."""
    Vm = np.asarray(snaps[f'track_v{layer_letter}'])[:, slot_indices]
    Va = np.asarray(snaps[f'track_v{layer_letter}_aux'])[:, slot_indices]
    n_steps = Vm.shape[0]
    perm_steps = _perm_steps(snaps)
    eps = 1e-12

    def gather(idx):
        if len(idx) == 0:
            return np.array([])
        return (np.abs(Vm[idx]) / (np.abs(Va[idx]) + eps)).ravel()

    early = np.arange(0, min(window, n_steps))
    if len(perm_steps) > 0:
        first = int(perm_steps[0])
        settled = np.arange(max(0, first - window), first)
        post = np.concatenate([np.arange(int(ps), min(int(ps) + window, n_steps))
                               for ps in perm_steps])
    else:
        settled = np.arange(max(0, n_steps - window), n_steps)
        post = np.array([], dtype=int)

    bm = float(snaps['momentum_main'])
    ba = float(snaps['momentum_aux'])
    noise_floor = _noise_ratio(bm, ba)

    fig = go.Figure()
    series = [('early',    gather(early),   '#1f77b4'),
              ('settled',  gather(settled), '#2ca02c'),
              ('post-perm', gather(post),   '#d62728')]
    for name, vals, color in series:
        if len(vals) == 0:
            continue
        log_vals = np.log10(np.maximum(vals, eps))
        fig.add_trace(go.Histogram(x=log_vals, name=name,
                                    marker_color=color, opacity=0.55,
                                    nbinsx=nbins))
    fig.add_vline(x=np.log10(noise_floor),
                  line=dict(color='gray', dash='dot'),
                  annotation_text=f'noise floor = {noise_floor:.2f}',
                  annotation_position='top')
    fig.add_vline(x=0.0,
                  line=dict(color='black', dash='dash'),
                  annotation_text='ratio = 1 (signal)',
                  annotation_position='top')
    fig.update_layout(
        title=f'{category_label}: log₁₀ |v_short|/|v_long| distribution',
        barmode='overlay',
        xaxis_title='log₁₀ ratio',
        yaxis_title='count',
        width=900, height=380,
    )
    return fig


def category_slots(track_info):
    """Return slot-index arrays per category for use with the plotters."""
    n_w = track_info['n_w1_within']
    n_c = track_info['n_w1_cross']
    n_2 = track_info['n_w2']
    return dict(
        w2_within=(np.arange(n_2),
                   list(map(tuple, track_info['track_w2_idx']))),
        w1_within=(np.arange(n_w),
                   list(map(tuple, track_info['track_w1_idx'][:n_w]))),
        w1_cross =(np.arange(n_w, n_w + n_c),
                   list(map(tuple, track_info['track_w1_idx'][n_w:]))),
    )

In [12]:
SLOTS = category_slots(TRACK)

# Zoom to a tight window around the single permutation event:
# 20 steps before to 200 steps after, raw per-step resolution.
_perm = SGD_CONFIG['permute_period']
STEP_WINDOW = (_perm - 20, _perm + 200)

plot_tracked_category(sgd_snaps, layer_letter='2',
                      slot_indices=SLOTS['w2_within'][0],
                      idx_pairs=SLOTS['w2_within'][1],
                      category_label='W2 (outgoing) — within-task',
                      downsample=1, step_window=STEP_WINDOW,
                      n_permuted_cols=TRACK['n_w2'] // 2).show()

plot_tracked_category(sgd_snaps, layer_letter='1',
                      slot_indices=SLOTS['w1_within'][0],
                      idx_pairs=SLOTS['w1_within'][1],
                      category_label='W1 (incoming) — within-task',
                      downsample=1, step_window=STEP_WINDOW,
                      n_permuted_cols=TRACK['n_w1_within'] // 2).show()

plot_tracked_category(sgd_snaps, layer_letter='1',
                      slot_indices=SLOTS['w1_cross'][0],
                      idx_pairs=SLOTS['w1_cross'][1],
                      category_label='W1 (incoming) — cross-task',
                      downsample=1, step_window=STEP_WINDOW,
                      n_permuted_cols=TRACK['n_w1_cross'] // 2).show()

# Aggregate ratio distributions across the three time windows. With
# permute_period=0 only `early` and `settled` are populated; set
# permute_period > 0 in SGD_CONFIG to get the post-perm series.
plot_post_perm_ratio_hist(sgd_snaps, layer_letter='2',
                          slot_indices=SLOTS['w2_within'][0],
                          category_label='W2 within').show()
plot_post_perm_ratio_hist(sgd_snaps, layer_letter='1',
                          slot_indices=SLOTS['w1_within'][0],
                          category_label='W1 within').show()
plot_post_perm_ratio_hist(sgd_snaps, layer_letter='1',
                          slot_indices=SLOTS['w1_cross'][0],
                          category_label='W1 cross').show()